In [ ]:
# --- Vishalam setup: find the code dataset among the inputs, unpack, install (no secrets) ---
import glob, os, shutil, subprocess, sys, tarfile, time
T0 = time.time()
def hours_left(limit=11.2):  # Kaggle stops sessions at 12 h; keep a margin for saving outputs
    return max(0.25, limit - (time.time() - T0) / 3600)
WORK, INPUT, CODE = "/kaggle/working", "/kaggle/input", "/tmp/vishalam"

def inputs_with(marker, hint=""):
    """Input directories that contain `marker` (a relative path) and whose path contains `hint`."""
    hits = glob.glob(f"{INPUT}/**/{marker}", recursive=True)
    dirs = [h[: -len(marker)].rstrip("/") for h in hits]
    return sorted(d for d in set(dirs) if hint in d)

if not os.path.isdir(CODE):
    tars = glob.glob(f"{INPUT}/**/vishalam_code.tar.gz", recursive=True)
    if tars:
        os.makedirs(CODE)
        with tarfile.open(tars[0]) as t:
            t.extractall(CODE, filter="data")
    else:  # Kaggle sometimes unpacks archives itself
        roots = [d for d in inputs_with("vishalam/tokenizer/bpe.py") if os.path.exists(f"{d}/requirements.txt")]
        if not roots:
            raise SystemExit("code dataset not attached - run tools/kaggle_driver.py push-code and re-push this kernel")
        shutil.copytree(roots[0], CODE)
os.chdir(CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("ready:", open("VERSION").read().strip() if os.path.exists("VERSION") else "code unpacked")


In [ ]:
# --- GPU guard: stop before spending quota if Kaggle gave us the wrong accelerator ---
import torch
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", names)
if not names or not all("T4" in n for n in names):
    raise SystemExit(f"Expected T4 GPUs, got {names}. Set Accelerator -> 'GPU T4 x2' and run again.")


In [ ]:
get_ipython().system("pip install -q uv && uv python install 3.11 && uv venv -q -p 3.11 /tmp/vllm_env "
                     "&& uv pip install -q -p /tmp/vllm_env/bin/python vllm==0.6.6.post1 transformers==4.46.3 setuptools")
VLLM_PY = "/tmp/vllm_env/bin/python"
import torch
os.makedirs(f"{WORK}/gen", exist_ok=True)
procs = [subprocess.Popen(f"CUDA_VISIBLE_DEVICES={g} {VLLM_PY} -m vishalam.data.agent.gen_tasks "
                          f"--out {WORK}/gen/cand_{g}.jsonl --n 100000 --seed {200 + g} "
                          f"--max-hours {hours_left() - 0.6:.2f} > {WORK}/gen/log_{g}.txt 2>&1", shell=True)
         for g in range(torch.cuda.device_count())]
while any(p.poll() is None for p in procs):
    time.sleep(900)
    for g in range(len(procs)):
        lines = [l for l in open(f"{WORK}/gen/log_{g}.txt").read().splitlines() if l.startswith("candidates")]
        print(f"gpu {g}:", lines[-1] if lines else "(loading)", flush=True)
for g in range(len(procs)):
    print(open(f"{WORK}/gen/log_{g}.txt").read()[-2000:])
